## INICIALIZAÇÃO

In [1]:
import os
import sys
import datetime
import pandas as pd
import gc #limpar cache

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

import random
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from models.nn.cnnbilstm import CNNBiLSTM
from models.nn.resnetlstm import ResNetLSTM
#from models.nn.resnetlstm import ResNetLSTM50
from preprocessing.loaders import build_mel_dataloader, build_video_dataloader
from preprocessing.balanced_dataset import balanced_df

In [2]:
print("Versão do Torch:", torch.__version__)
print("Versão do Torchvision:", torchvision.__version__)
print(f"CUDA disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Versão do CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

seed = 435
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Semente: {seed}")
print(f"Dispositivo: {device}")

Versão do Torch: 2.5.1+cu121
Versão do Torchvision: 0.20.1+cu121
CUDA disponível: True
Versão do CUDA: 12.1
GPU: NVIDIA RTX A4500
Semente: 435
Dispositivo: cuda


In [4]:
CSV_PATH = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"

train_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

valid_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

train_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 36304/37620 exemplos válidos.
Dataset: 11224/12540 exemplos válidos.
Dataset: 37620/37620 exemplos válidos.
Dataset: 12540/12540 exemplos válidos.


## AVALIAÇÃO

#### Calcular o CCC

Para validar se a função gerada pela rede é boa, a forma ideal de comparar com o seu Ground Truth (que também deve ser mapeado para o tempo) é usando o CCC, que mede a concordância de trajetórias temporais:

$$ρ_c​= \dfrac{2ρσ_x​σ_y}{σ_x^2​+σ_y^2+(μ_x​−μ_y​)^2}​​$$

Onde ρ é a correlação de Pearson, μ é a média e σ2 é a variância. O CCC pune o modelo não apenas se a curva subir e descer nos momentos errados, mas também se houver um deslocamento de escala (ex: o modelo captou a variação, mas os valores absolutos estão muito baixos).

In [3]:
def calculate_ccc(y_true, y_pred):
    """Calcula o Concordance Correlation Coefficient (CCC)."""
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)

    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    numerator = 2 * covariance
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2

    return 0.0 if denominator == 0 else numerator / denominator

# TREINAMENTO

In [4]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    epochs=10,
    device="cuda",
    patience=5,
    checkpoint_path="best_model.pth",
    nome_modelo='ESQUECEU_O_NOME'
):
    """
    Treina a rede e a avalia a cada época usando o CCC como métrica principal.

    Argumentos:
        model:            Modelo PyTorch.
        train_loader:     DataLoader para os dados de treino.
        val_loader:       DataLoader para os dados de validação.
        optimizer:        Otimizador (ex: AdamW).
        criterion:        Função de perda (ex: nn.MSELoss()).
        scheduler:        Scheduler de learning rate (ex: ReduceLROnPlateau).
        epochs:           Número máximo de épocas.
        device:           Dispositivo de treino ("cuda" ou "cpu").
        patience:         Épocas sem melhora no CCC antes de acionar early stopping.
        checkpoint_path:  Caminho para salvar o melhor modelo.
    """

    assert nome_modelo != 'ESQUECEU_O_NOME', "Coloque um nome nesse modelo, por favor"

    model.to(device)
    best_val_ccc = -1.0 # CCC varia de - 1 a 1, quanto mais perto de 1, melhor
    epochs_no_improve = 0

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join("runs", f"arousal_{current_time}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):

        # ------------------------------------------------------------------ #
        # TREINO
        # ------------------------------------------------------------------ #
        model.train()
        train_loss = 0.0

        for videos, masks, targets in tqdm(
            train_loader, desc=f"Época {epoch+1}/{epochs} [treino]"
        ):
            videos = videos.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos).squeeze(1)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * videos.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------------------------ #
        # VALIDAÇÃO
        # ------------------------------------------------------------------ #
        model.eval()
        val_loss = 0.0
        all_true = []
        all_pred = []

        with torch.no_grad():
            for videos, masks, targets in tqdm(
                val_loader, desc=f"Época {epoch+1}/{epochs} [val]"
            ):
                videos = videos.to(device)
                targets = targets.to(device)

                outputs = model(videos).squeeze(1)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * videos.size(0)

                all_true.extend(targets.cpu().numpy())
                all_pred.extend(outputs.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(all_true) - np.array(all_pred)))
        val_ccc = calculate_ccc(np.array(all_true), np.array(all_pred))

        scheduler.step(val_loss)

        # ------------------------------------------------------------------ #
        # TENSORBOARD
        # ------------------------------------------------------------------ #
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE", val_mae, epoch)
        writer.add_scalar("Metrics/CCC", val_ccc, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Época [{epoch+1}/{epochs}]"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Val MAE: {val_mae:.4f}"
            f" | Val CCC: {val_ccc:.4f}"
            f" | LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ------------------------------------------------------------------ #
        # CHECKPOINT + EARLY STOPPING
        # ------------------------------------------------------------------ #
        if val_ccc > best_val_ccc:
            best_val_ccc = val_ccc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_ccc": best_val_ccc,
            }, checkpoint_path)
            print(f"Novo melhor modelo salvo! (CCC: {best_val_ccc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"Sem melhora por {epochs_no_improve}/{patience} épocas.")
            if epochs_no_improve >= patience:
                print("Early stopping ativado.")
                break

    writer.close()
    print(f"\nTreinamento concluído. Melhor CCC: {best_val_ccc:.4f}")

# TREINAMENTO C/ FINE-TUNING

In [5]:
def train_model_fine_tuning(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    epochs=10,
    device="cuda",
    patience=5,
    checkpoint_path="best_model.pth",
    unfreeze_epoch=3,
    nome_modelo='ESQUECEU_O_NOME'
):
    """
    Treina a rede e a avalia a cada época usando o CCC como métrica principal.
    Congela a ResNet inicial até atingir a 'unfreeze_epoch'.
    """

    assert nome_modelo != 'ESQUECEU_O_NOME', "Erro: Coloque um nome nesse modelo, por favor"

    assert not os.path.exists(checkpoint_path), f"Erro: O arquivo de checkpoint '{checkpoint_path}' já existe. Mude o nome para não sobrescrever."

    model.to(device)
    best_val_ccc = -1.0 
    epochs_no_improve = 0

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join("runs", f"arousal_{nome_modelo}_{current_time}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    # ================================================================== #
    # CONFIGURAÇÃO INICIAL: CONGELAR Aoutputs, targets RESNET
    # ================================================================== #
    # Congela toda a ResNet, EXCETO a primeira camada de 1 canal que você alterou
    print("=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...")
    for name, param in model.cnn.named_parameters():
        if "0.weight" not in name:  # Mantém a primeira camada aberta pois ela foi reinicializada do zero
            param.requires_grad = False
            
    # Força o otimizador a olhar apenas para os parâmetros que estão ativos no momento
    # (Evita que o otimizador gaste memória/etapas com gradientes congelados)
    optimizer.param_groups[0]['params'] = [p for p in model.parameters() if p.requires_grad]

    for epoch in range(epochs):

        # ================================================================== #
        # LÓGICA DE DESCONGELAMENTO (UNFREEZE) CORRIGIDA
        # ================================================================== #
        if epoch == unfreeze_epoch:
            print(f"\n[ÉPOCA {epoch+1}] => ATENÇÃO: Descongelando os pesos da ResNet para Fine-Tuning completo!")
            
            # 1. Ativa o gradiente para absolutamente todos os parâmetros do modelo
            for param in model.parameters():
                param.requires_grad = True
                
            # 2. Captura a Learning Rate atualizada pelo scheduler antes de resetar
            current_lr = optimizer.param_groups[0]['lr']
            
            # 3. Recria o otimizador passando todos os parâmetros e preservando a LR e tipo de otimizador
            # (Exemplo assumindo que você usa AdamW, ajuste para o seu otimizador se for Adam ou SGD)
            optimizer = type(optimizer)(model.parameters(), lr=current_lr, weight_decay=optimizer.param_groups[0].get('weight_decay', 0))

        # ------------------------------------------------------------------ #
        # TREINO
        # ------------------------------------------------------------------ #
        model.train()
        train_loss = 0.0

        for videos, masks, targets in tqdm(
            train_loader, desc=f"Época {epoch+1}/{epochs} [treino]"
        ):
            videos = videos.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos).squeeze(1)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * videos.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------------------------ #
        # VALIDAÇÃO
        # ------------------------------------------------------------------ #
        model.eval()
        val_loss = 0.0
        all_true = []
        all_pred = []

        with torch.no_grad():
            for videos, masks, targets in tqdm(
                val_loader, desc=f"Época {epoch+1}/{epochs} [val]"
            ):
                videos = videos.to(device)
                targets = targets.to(device)

                outputs = model(videos).squeeze(1)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * videos.size(0)

                all_true.extend(targets.cpu().numpy())
                all_pred.extend(outputs.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(all_true) - np.array(all_pred)))
        val_ccc = calculate_ccc(np.array(all_true), np.array(all_pred))

        scheduler.step(val_loss)

        # ------------------------------------------------------------------ #
        # TENSORBOARD
        # ------------------------------------------------------------------ #
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE", val_mae, epoch)
        writer.add_scalar("Metrics/CCC", val_ccc, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        # Print explicativo do estado atual do treino
        status_resnet = "Congelada" if epoch < unfreeze_epoch else "Aberta (Fine-Tuning)"
        print(
            f"Época [{epoch+1}/{epochs}] [{status_resnet}]"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Val MAE: {val_mae:.4f}"
            f" | Val CCC: {val_ccc:.4f}"
            f" | LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ------------------------------------------------------------------ #
        # CHECKPOINT + EARLY STOPPING
        # ------------------------------------------------------------------ #
        if val_ccc > best_val_ccc:
            best_val_ccc = val_ccc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_ccc": best_val_ccc,
            }, checkpoint_path)
            print(f"Novo melhor modelo salvo! (CCC: {best_val_ccc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"Sem melhora por {epochs_no_improve}/{patience} épocas.")
            if epochs_no_improve >= patience:
                print("Early stopping ativado.")
                break
        
        del all_true, all_pred  # Deleta as listas de cpu que acumularam dados
        torch.cuda.empty_cache() # Libera o cache oculto da GPU
        gc.collect()
    
    writer.close()
    print(f"\nTreinamento concluído. Melhor CCC: {best_val_ccc:.4f}")

In [8]:
model = CNNBiLSTM(
    in_channels=1,
    frame_step=15,
    use_dropout=True,
    use_batch_norm=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

os.makedirs("/mnt/storage_C4/gaussian_football/models/checkpoints", exist_ok=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Parâmetros treináveis: 11,472,705


In [9]:
train_model(
    model=model,
    train_loader=train_video_loader,
    val_loader=valid_video_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=10,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/best_model.pth",
)

TensorBoard: runs/arousal_20260605-182452


Época 1/10 [val]: 100%|██████████| 3135/3135 [24:59<00:00,  2.09it/s]


Época [1/10] | Train Loss: 0.0258 | Val Loss: 0.0211 | Val MAE: 0.0339 | Val CCC: 0.0101 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0101)


Época 2/10 [val]: 100%|██████████| 3135/3135 [24:43<00:00,  2.11it/s]


Época [2/10] | Train Loss: 0.0161 | Val Loss: 0.0210 | Val MAE: 0.0304 | Val CCC: 0.0024 | LR: 1.00e-04
Sem melhora por 1/5 épocas.


...

Época [4/10] | Train Loss: 0.0160 | Val Loss: 0.0208 | Val MAE: 0.0380 | Val CCC: -0.0006 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 5/10 [val]: 100%|██████████| 3135/3135 [24:58<00:00,  2.09it/s]


Época [5/10] | Train Loss: 0.0160 | Val Loss: 0.0206 | Val MAE: 0.0379 | Val CCC: 0.0045 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 6/10 [val]: 100%|██████████| 3135/3135 [25:01<00:00,  2.09it/s]

Época [6/10] | Train Loss: 0.0160 | Val Loss: 0.0208 | Val MAE: 0.0379 | Val CCC: -0.0038 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.0101


    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
    patience=5,
    epochs=15,

In [7]:
model = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

train_model(
    model=model,
    train_loader=train_video_loader,
    val_loader=valid_video_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=15,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm.pth",
)

Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_20260608-155047


Época 1/15 [val]: 100%|██████████| 3135/3135 [25:26<00:00,  2.05it/s]


Época [1/15] | Train Loss: 0.0160 | Val Loss: 0.0205 | Val MAE: 0.0482 | Val CCC: -0.0004 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: -0.0004)


Época 2/15 [val]: 100%|██████████| 3135/3135 [25:30<00:00,  2.05it/s]


Época [2/15] | Train Loss: 0.0159 | Val Loss: 0.0205 | Val MAE: 0.0406 | Val CCC: 0.0162 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0162)


...

Época [5/15] | Train Loss: 0.0159 | Val Loss: 0.0203 | Val MAE: 0.0517 | Val CCC: 0.0128 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 6/15 [val]: 100%|██████████| 3135/3135 [25:45<00:00,  2.03it/s]


Época [6/15] | Train Loss: 0.0159 | Val Loss: 0.0203 | Val MAE: 0.0458 | Val CCC: 0.0156 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 7/15 [val]: 100%|██████████| 3135/3135 [25:39<00:00,  2.04it/s]

Época [7/15] | Train Loss: 0.0158 | Val Loss: 0.0203 | Val MAE: 0.0473 | Val CCC: 0.0158 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.0162


## Treino com dados balanceados

Com a finalidade de avaliar o treinamento do modelo com um dataset balanceado (50% de clips highlight e 50% não-highlights), vamos filtrar o conjunto de treino com base na quantidade de clips highlight em uma partida e adicionando apenas a quantidade correspondente de clips não-highlights.

Inicialmente o threshold para o arousal_score para que o clip ser=ja considerado highlight é 0.5. 

In [6]:
# balanceando os dados

all_data = pd.read_csv("/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv")

all_train = all_data[all_data['split'] == 'train'] # todos os dados do conjunto de treino
all_valid = all_data[all_data['split'] == 'valid']
all_test = all_data[all_data['split'] == 'test']

balanced_train = balanced_df(all_train, 'game_id', threshold=0.5, random_state=seed)
balanced_valid = balanced_df(all_valid, 'game_id', threshold=0.5, random_state=seed)
balanced_test = balanced_df(all_test, 'game_id', threshold=0.5, random_state=seed)

# salvar em csv
balanced_train.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_train.csv', index=False)
balanced_valid.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_val.csv', index=False)
balanced_test.to_csv('/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_test.csv', index=False)

In [7]:
# criação dos loaders:
TRAIN_PATH = '/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_train.csv'
VAL_PATH = '/mnt/storage_C4/gaussian_football/data/labels/balanced_labels_val.csv'

train_mel_loader_balanced = build_mel_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

valid_mel_loader_balanced = build_mel_dataloader(
    csv_path=VAL_PATH,
    split="valid",
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

train_video_loader_balanced = build_video_dataloader(
    csv_path=TRAIN_PATH,
    split="train",
    batch_size=2,
    shuffle=True,
    num_workers=0,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader_balanced = build_video_dataloader(
    csv_path=VAL_PATH,
    split="valid",
    batch_size=2,
    shuffle=False,
    num_workers=0,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 1467/1514 exemplos válidos.
Dataset: 579/654 exemplos válidos.
Dataset: 1514/1514 exemplos válidos.
Dataset: 654/654 exemplos válidos.


O volume do dataset diminui consideravelmente.

Antes do balanceamento:
- Dataset train mel: 36304/37620 exemplos válidos.
- Dataset val mel: 11224/12540 exemplos válidos.
- Dataset train video: 37620/37620 exemplos válidos.
- Dataset val video: 12540/12540 exemplos válidos.

Depois do balanceamento:
- Dataset train mel: 1467/1514 exemplos válidos.
- Dataset val mel: 579/654 exemplos válidos.
- Dataset train video: 1514/1514 exemplos válidos.
- Dataset val video: 654/654 exemplos válidos.

### Treino da CNNiLSTM

In [10]:
model_balanced = CNNBiLSTM(
    in_channels=1,
    frame_step=15,
    use_dropout=True,
    use_batch_norm=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

os.makedirs("/mnt/storage_C4/gaussian_football/models/checkpoints", exist_ok=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in model_balanced.parameters() if p.requires_grad):,}")

Parâmetros treináveis: 11,472,705


In [11]:
train_model(
    model=model_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=10,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/best_model_balanced.pth",
)

TensorBoard: runs/arousal_20260609-112747


Época 1/10 [treino]:   0%|          | 0/379 [00:00<?, ?it/s]

Época 1/10 [val]: 100%|██████████| 164/164 [03:11<00:00,  1.17s/it]


Época [1/10] | Train Loss: 0.2967 | Val Loss: 0.2120 | Val MAE: 0.4110 | Val CCC: 0.0640 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0640)


Época 2/10 [val]: 100%|██████████| 164/164 [03:11<00:00,  1.17s/it]


...

Época [8/10] | Train Loss: 0.2263 | Val Loss: 0.2240 | Val MAE: 0.4272 | Val CCC: -0.0035 | LR: 5.00e-05
Sem melhora por 1/5 épocas.


Época 9/10 [val]: 100%|██████████| 164/164 [03:20<00:00,  1.22s/it]


Época [9/10] | Train Loss: 0.2182 | Val Loss: 0.2021 | Val MAE: 0.4150 | Val CCC: 0.0555 | LR: 5.00e-05
Sem melhora por 2/5 épocas.


Época 10/10 [val]: 100%|██████████| 164/164 [03:17<00:00,  1.21s/it]

Época [10/10] | Train Loss: 0.2163 | Val Loss: 0.2062 | Val MAE: 0.4112 | Val CCC: 0.0539 | LR: 5.00e-05
Sem melhora por 3/5 épocas.

Treinamento concluído. Melhor CCC: 0.0825


### Treino da ResNetLSTM com dados balanceados

In [12]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=15,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced.pth",
)

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_20260609-133747


Época 1/15 [val]: 100%|██████████| 164/164 [02:59<00:00,  1.10s/it]


Época [1/15] | Train Loss: 0.1950 | Val Loss: 0.1805 | Val MAE: 0.3814 | Val CCC: 0.1897 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1897)


Época 2/15 [val]: 100%|██████████| 164/164 [02:56<00:00,  1.08s/it]


...

Época [13/15] | Train Loss: 0.0225 | Val Loss: 0.2050 | Val MAE: 0.3293 | Val CCC: 0.3393 | LR: 2.50e-05
Sem melhora por 3/5 épocas.


Época 14/15 [val]: 100%|██████████| 164/164 [02:56<00:00,  1.08s/it]


Época [14/15] | Train Loss: 0.0166 | Val Loss: 0.2018 | Val MAE: 0.3352 | Val CCC: 0.3292 | LR: 2.50e-05
Sem melhora por 4/5 épocas.


Época 15/15 [val]: 100%|██████████| 164/164 [02:53<00:00,  1.06s/it]

Época [15/15] | Train Loss: 0.0125 | Val Loss: 0.1989 | Val MAE: 0.3284 | Val CCC: 0.3617 | LR: 2.50e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3729


### Treino da ResNetLSTM com dados balanceados COM fine-tuning (congelamento de 5 epocas)

In [17]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced_finetuning.pth",
    unfreeze_epoch=5,
    nome_modelo='RESNET_FINETUNING_1A_TENTATIVA'
)

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_RESNET_FINETUNING_1A_TENTATIVA_20260609-163404
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 164/164 [02:53<00:00,  1.06s/it]


Época [1/20] [Congelada] | Train Loss: 0.1938 | Val Loss: 0.1735 | Val MAE: 0.3754 | Val CCC: 0.2346 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2346)


Época 2/20 [val]: 100%|██████████| 164/164 [02:50<00:00,  1.04s/it]


...

Época [13/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0387 | Val Loss: 0.1698 | Val MAE: 0.2952 | Val CCC: 0.4557 | LR: 5.00e-05
Sem melhora por 3/5 épocas.


Época 14/20 [val]: 100%|██████████| 164/164 [02:53<00:00,  1.06s/it]


Época [14/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0319 | Val Loss: 0.1784 | Val MAE: 0.2971 | Val CCC: 0.4242 | LR: 2.50e-05
Sem melhora por 4/5 épocas.


Época 15/20 [val]: 100%|██████████| 164/164 [02:52<00:00,  1.05s/it]

Época [15/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0207 | Val Loss: 0.1777 | Val MAE: 0.3052 | Val CCC: 0.4123 | LR: 2.50e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4720


### Treino da ResNetLSTM com dados balanceados COM fine-tuning (congelamento de 3 épocas)

In [18]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced_finetuning.pth",
    unfreeze_epoch=3,
    nome_modelo='RESNET_FINETUNING_1A_TENTATIVA_3EPOCHS'
)

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_RESNET_FINETUNING_1A_TENTATIVA_3EPOCHS_20260610-142329
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 164/164 [03:02<00:00,  1.11s/it]


Época [1/20] [Congelada] | Train Loss: 0.1980 | Val Loss: 0.1699 | Val MAE: 0.3882 | Val CCC: 0.1684 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1684)


Época 2/20 [val]: 100%|██████████| 164/164 [03:05<00:00,  1.13s/it]


...

Época [13/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0213 | Val Loss: 0.1865 | Val MAE: 0.3084 | Val CCC: 0.4013 | LR: 2.50e-05
Sem melhora por 3/5 épocas.


Época 14/20 [val]: 100%|██████████| 164/164 [03:04<00:00,  1.12s/it]


Época [14/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0193 | Val Loss: 0.1940 | Val MAE: 0.3397 | Val CCC: 0.3199 | LR: 2.50e-05
Sem melhora por 4/5 épocas.


Época 15/20 [val]: 100%|██████████| 164/164 [03:04<00:00,  1.12s/it]

Época [15/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0160 | Val Loss: 0.1921 | Val MAE: 0.3158 | Val CCC: 0.3693 | LR: 2.50e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4337


### Treino da ResNetLSTM com dados balanceados COM fine-tuning (congelamento de 4 épocas)

In [19]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced_finetuning.pth",
    unfreeze_epoch=3,
    nome_modelo='RESNET_FINETUNING_1A_TENTATIVA_4EPOCHS'
)

Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_RESNET_FINETUNING_1A_TENTATIVA_4EPOCHS_20260610-171536
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 164/164 [03:03<00:00,  1.12s/it]


Época [1/20] [Congelada] | Train Loss: 0.1937 | Val Loss: 0.1703 | Val MAE: 0.3768 | Val CCC: 0.2297 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2297)


Época 2/20 [val]: 100%|██████████| 164/164 [03:05<00:00,  1.13s/it]


Época [2/20] [Congelada] | Train Loss: 0.1779 | Val Loss: 0.2689 | Val MAE: 0.3908 | Val CCC: 0.1125 | LR: 1.00e-04
Sem melhora por 1/5 épocas.


...

Época [14/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0282 | Val Loss: 0.1889 | Val MAE: 0.3157 | Val CCC: 0.4014 | LR: 2.50e-05
Sem melhora por 3/5 épocas.


Época 15/20 [val]: 100%|██████████| 164/164 [03:03<00:00,  1.12s/it]


Época [15/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0206 | Val Loss: 0.1881 | Val MAE: 0.3125 | Val CCC: 0.3866 | LR: 2.50e-05
Sem melhora por 4/5 épocas.


Época 16/20 [val]: 100%|██████████| 164/164 [03:03<00:00,  1.12s/it]

Época [16/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0154 | Val Loss: 0.1883 | Val MAE: 0.3183 | Val CCC: 0.3710 | LR: 2.50e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4252


### Treino da ResNetLSTM com dados balanceados COM fine-tuning (congelamento de 6 épocas)

In [20]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced_finetuning.pth",
    unfreeze_epoch=3,
    nome_modelo='RESNET_FINETUNING_1A_TENTATIVA_6EPOCHS'
)

Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_RESNET_FINETUNING_1A_TENTATIVA_6EPOCHS_20260610-202116
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 164/164 [03:04<00:00,  1.12s/it]


Época [1/20] [Congelada] | Train Loss: 0.1982 | Val Loss: 0.1806 | Val MAE: 0.3898 | Val CCC: 0.1751 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1751)


Época 2/20 [val]: 100%|██████████| 164/164 [03:04<00:00,  1.13s/it]


Época [2/20] [Congelada] | Train Loss: 0.1846 | Val Loss: 0.1679 | Val MAE: 0.3678 | Val CCC: 0.2566 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2566)


...

Época [15/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0273 | Val Loss: 0.2034 | Val MAE: 0.3500 | Val CCC: 0.3010 | LR: 5.00e-05
Sem melhora por 3/5 épocas.


Época 16/20 [val]: 100%|██████████| 164/164 [02:57<00:00,  1.08s/it]


Época [16/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0205 | Val Loss: 0.1931 | Val MAE: 0.3228 | Val CCC: 0.3703 | LR: 5.00e-05
Sem melhora por 4/5 épocas.


Época 17/20 [val]: 100%|██████████| 164/164 [02:57<00:00,  1.08s/it]

Época [17/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0183 | Val Loss: 0.1959 | Val MAE: 0.3295 | Val CCC: 0.3517 | LR: 2.50e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3773


### Treino da ResNetLSTM com dados balanceados COM fine-tuning (congelamento de 7 épocas)

In [21]:
resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm_balanced_finetuning.pth",
    unfreeze_epoch=3,
    nome_modelo='RESNET_FINETUNING_1A_TENTATIVA_7EPOCHS'
)

Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_RESNET_FINETUNING_1A_TENTATIVA_7EPOCHS_20260610-233427
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 164/164 [02:58<00:00,  1.09s/it]


Época [1/20] [Congelada] | Train Loss: 0.1985 | Val Loss: 0.1771 | Val MAE: 0.3942 | Val CCC: 0.1390 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1390)


Época 2/20 [val]: 100%|██████████| 164/164 [03:01<00:00,  1.10s/it]


Época [2/20] [Congelada] | Train Loss: 0.1838 | Val Loss: 0.1969 | Val MAE: 0.3905 | Val CCC: 0.1466 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1466)


...

Época [12/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1064 | Val Loss: 0.1665 | Val MAE: 0.3224 | Val CCC: 0.4091 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 13/20 [val]: 100%|██████████| 164/164 [02:58<00:00,  1.09s/it]


Época [13/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0743 | Val Loss: 0.2135 | Val MAE: 0.3502 | Val CCC: 0.3056 | LR: 5.00e-05
Sem melhora por 4/5 épocas.


Época 14/20 [val]: 100%|██████████| 164/164 [02:57<00:00,  1.08s/it]

Época [14/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0440 | Val Loss: 0.2020 | Val MAE: 0.3368 | Val CCC: 0.3338 | LR: 5.00e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4148


### Treino da ResNetLSTM50 com dados balanceados (CONGELAMENTO DE 5 EPOCAS) (BATCH SIZE = 2)

In [7]:
# BATCH_SIZE FOI MUDADO DE 4 PRA 2 POR ERRO DE MEMÓRIA

resnet_balanced = ResNetLSTM50(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning.pth",
    unfreeze_epoch=5,
    nome_modelo='RESNET50_FINETUNING_CONGELAMENTO_5_EPOCAS'
)

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Parâmetros treináveis: 29,867,201
TensorBoard: runs/arousal_RESNET50_FINETUNING_CONGELAMENTO_5_EPOCAS_20260611-112037
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [03:19<00:00,  1.64it/s]


Época [1/20] [Congelada] | Train Loss: 0.2066 | Val Loss: 0.1751 | Val MAE: 0.3850 | Val CCC: 0.1890 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1890)


Época 2/20 [val]: 100%|██████████| 327/327 [03:20<00:00,  1.63it/s]


...

Época [8/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1616 | Val Loss: 0.3534 | Val MAE: 0.4732 | Val CCC: 0.1817 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 9/20 [val]: 100%|██████████| 327/327 [03:21<00:00,  1.63it/s]


Época [9/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1422 | Val Loss: 0.2165 | Val MAE: 0.3608 | Val CCC: 0.3292 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 10/20 [val]: 100%|██████████| 327/327 [03:22<00:00,  1.61it/s]

Época [10/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1088 | Val Loss: 0.1883 | Val MAE: 0.3369 | Val CCC: 0.3650 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4245


### Treino da ResNetLSTM50 com dados balanceados (CONGELAMENTO DE 3 EPOCAS)

In [10]:
# BATCH_SIZE FOI MUDADO DE 4 PRA 2 POR ERRO DE MEMÓRIA

resnet_balanced = ResNetLSTM50(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_3_epochs_bs2.pth",
    unfreeze_epoch=3,
    nome_modelo='RESNET50_FINETUNING_CONGELAMENTO_3_EPOCAS'
)

Parâmetros treináveis: 29,867,201
TensorBoard: runs/arousal_RESNET50_FINETUNING_CONGELAMENTO_3_EPOCAS_20260611-144148
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [03:23<00:00,  1.61it/s]


Época [1/20] [Congelada] | Train Loss: 0.2017 | Val Loss: 0.1803 | Val MAE: 0.3857 | Val CCC: 0.1709 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1709)


Época 2/20 [val]: 100%|██████████| 327/327 [03:24<00:00,  1.60it/s]


Época [2/20] [Congelada] | Train Loss: 0.1883 | Val Loss: 0.1768 | Val MAE: 0.3658 | Val CCC: 0.2535 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2535)


...

Época [18/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0268 | Val Loss: 0.2442 | Val MAE: 0.3612 | Val CCC: 0.2929 | LR: 1.00e-04
Sem melhora por 2/5 épocas.


Época 19/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]


Época [19/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0266 | Val Loss: 0.2262 | Val MAE: 0.3464 | Val CCC: 0.2988 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 20/20 [val]: 100%|██████████| 327/327 [03:26<00:00,  1.59it/s]

Época [20/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0243 | Val Loss: 0.2425 | Val MAE: 0.3530 | Val CCC: 0.2518 | LR: 1.00e-04
Sem melhora por 4/5 épocas.

Treinamento concluído. Melhor CCC: 0.3713


### Treino da ResNetLSTM50 com dados balanceados (CONGELAMENTO DE 4 EPOCAS)

In [11]:
# BATCH_SIZE FOI MUDADO DE 4 PRA 2 POR ERRO DE MEMÓRIA

resnet_balanced = ResNetLSTM50(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_4_epochs.pth",
    unfreeze_epoch=4,
    nome_modelo='RESNET50_FINETUNING_CONGELAMENTO_4_EPOCAS'
)

Parâmetros treináveis: 29,867,201
TensorBoard: runs/arousal_RESNET50_FINETUNING_CONGELAMENTO_4_EPOCAS_20260611-194436
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [03:26<00:00,  1.58it/s]


Época [1/20] [Congelada] | Train Loss: 0.1996 | Val Loss: 0.1782 | Val MAE: 0.3664 | Val CCC: 0.2629 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2629)


Época 2/20 [val]: 100%|██████████| 327/327 [03:21<00:00,  1.62it/s]


Época [2/20] [Congelada] | Train Loss: 0.1883 | Val Loss: 0.2072 | Val MAE: 0.3845 | Val CCC: 0.2534 | LR: 1.00e-04
Sem melhora por 1/5 épocas.


...

Época [7/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1744 | Val Loss: 0.1992 | Val MAE: 0.3430 | Val CCC: 0.3451 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 8/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]


Época [8/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1668 | Val Loss: 0.1908 | Val MAE: 0.3642 | Val CCC: 0.2893 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 9/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]

Época [9/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1581 | Val Loss: 0.2205 | Val MAE: 0.3850 | Val CCC: 0.2914 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3675


### Treino da ResNetLSTM50 com dados balanceados (CONGELAMENTO DE 6 EPOCAS)

In [12]:
# BATCH_SIZE FOI MUDADO DE 4 PRA 2 POR ERRO DE MEMÓRIA

resnet_balanced = ResNetLSTM50(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_6_epochs.pth",
    unfreeze_epoch=6,
    nome_modelo='RESNET50_FINETUNING_CONGELAMENTO_6_EPOCAS'
)

Parâmetros treináveis: 29,867,201
TensorBoard: runs/arousal_RESNET50_FINETUNING_CONGELAMENTO_6_EPOCAS_20260611-215838
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]


Época [1/20] [Congelada] | Train Loss: 0.1990 | Val Loss: 0.1638 | Val MAE: 0.3710 | Val CCC: 0.2466 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2466)


Época 2/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]


Época [2/20] [Congelada] | Train Loss: 0.1853 | Val Loss: 0.1664 | Val MAE: 0.3633 | Val CCC: 0.2691 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2691)


...

Época [9/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0914 | Val Loss: 0.2829 | Val MAE: 0.4057 | Val CCC: 0.0733 | LR: 5.00e-05
Sem melhora por 3/5 épocas.


Época 10/20 [val]: 100%|██████████| 327/327 [03:26<00:00,  1.58it/s]


Época [10/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0699 | Val Loss: 0.2127 | Val MAE: 0.3596 | Val CCC: 0.3096 | LR: 5.00e-05
Sem melhora por 4/5 épocas.


Época 11/20 [val]: 100%|██████████| 327/327 [03:26<00:00,  1.59it/s]

Época [11/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0476 | Val Loss: 0.2010 | Val MAE: 0.3322 | Val CCC: 0.3483 | LR: 5.00e-05
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4138


### Treino da ResNetLSTM50 com dados balanceados (CONGELAMENTO DE 7 EPOCAS)

In [13]:
# BATCH_SIZE FOI MUDADO DE 4 PRA 2 POR ERRO DE MEMÓRIA

resnet_balanced = ResNetLSTM50(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_7_epochs.pth",
    unfreeze_epoch=7,
    nome_modelo='RESNET50_FINETUNING_CONGELAMENTO_7_EPOCAS'
)

Parâmetros treináveis: 29,867,201
TensorBoard: runs/arousal_RESNET50_FINETUNING_CONGELAMENTO_7_EPOCAS_20260612-004105
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [03:25<00:00,  1.59it/s]


Época [1/20] [Congelada] | Train Loss: 0.2019 | Val Loss: 0.1801 | Val MAE: 0.3732 | Val CCC: 0.2225 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2225)


Época 2/20 [val]: 100%|██████████| 327/327 [03:22<00:00,  1.61it/s]


Época [2/20] [Congelada] | Train Loss: 0.1978 | Val Loss: 0.1840 | Val MAE: 0.3820 | Val CCC: 0.2301 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2301)


...

Época [8/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1892 | Val Loss: 0.1922 | Val MAE: 0.3631 | Val CCC: 0.2509 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 9/20 [val]: 100%|██████████| 327/327 [03:23<00:00,  1.60it/s]


Época [9/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1805 | Val Loss: 0.1816 | Val MAE: 0.3477 | Val CCC: 0.3123 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 10/20 [val]: 100%|██████████| 327/327 [03:24<00:00,  1.60it/s]

Época [10/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1724 | Val Loss: 0.1981 | Val MAE: 0.3547 | Val CCC: 0.3425 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3458


## TREINAR MUDANDO NUM_LAYERS (PATIENCE = 10)

NUM_LAYERS = 1

In [9]:
NUM_LAYERS = 1

resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=NUM_LAYERS,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=10,
    checkpoint_path=f"/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10.pth",
    unfreeze_epoch=7,
    nome_modelo=f'RESNET50_FINETUNING_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10'
)

Parâmetros treináveis: 12,812,993
TensorBoard: runs/arousal_RESNET50_FINETUNING_NUM_LAYERS_1_PATIENCE_10_20260613-090439
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [02:54<00:00,  1.87it/s]


Época [1/20] [Congelada] | Train Loss: 0.2064 | Val Loss: 0.1747 | Val MAE: 0.3775 | Val CCC: 0.2334 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2334)


Época 2/20 [val]: 100%|██████████| 327/327 [02:53<00:00,  1.88it/s]


Época [2/20] [Congelada] | Train Loss: 0.1931 | Val Loss: 0.1606 | Val MAE: 0.3528 | Val CCC: 0.3122 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.3122)


...

Época [17/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0239 | Val Loss: 0.2503 | Val MAE: 0.3764 | Val CCC: 0.1910 | LR: 5.00e-05
Sem melhora por 8/10 épocas.


Época 18/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.86it/s]


Época [18/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0202 | Val Loss: 0.2415 | Val MAE: 0.3846 | Val CCC: 0.1851 | LR: 5.00e-05
Sem melhora por 9/10 épocas.


Época 19/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.86it/s]

Época [19/20] [Aberta (Fine-Tuning)] | Train Loss: 0.0182 | Val Loss: 0.2496 | Val MAE: 0.3829 | Val CCC: 0.1850 | LR: 5.00e-05
Sem melhora por 10/10 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.4167


NUM_LAYERS = 3

In [10]:
NUM_LAYERS = 3

resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=NUM_LAYERS,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=10,
    checkpoint_path=f"/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10.pth",
    unfreeze_epoch=7,
    nome_modelo=f'RESNET50_FINETUNING_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10'
)

Parâmetros treináveis: 15,966,913
TensorBoard: runs/arousal_RESNET50_FINETUNING_NUM_LAYERS_3_PATIENCE_10_20260613-122947
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.86it/s]


Época [1/20] [Congelada] | Train Loss: 0.2023 | Val Loss: 0.1835 | Val MAE: 0.3735 | Val CCC: 0.2474 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2474)


Época 2/20 [val]: 100%|██████████| 327/327 [02:53<00:00,  1.89it/s]


Época [2/20] [Congelada] | Train Loss: 0.1942 | Val Loss: 0.1768 | Val MAE: 0.3588 | Val CCC: 0.2826 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2826)


...

Época [15/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1775 | Val Loss: 0.1757 | Val MAE: 0.3585 | Val CCC: 0.2704 | LR: 1.00e-04
Sem melhora por 8/10 épocas.


Época 16/20 [val]: 100%|██████████| 327/327 [02:50<00:00,  1.92it/s]


Época [16/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1777 | Val Loss: 0.1762 | Val MAE: 0.3746 | Val CCC: 0.2271 | LR: 1.00e-04
Sem melhora por 9/10 épocas.


Época 17/20 [val]: 100%|██████████| 327/327 [02:53<00:00,  1.88it/s]

Época [17/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1749 | Val Loss: 0.1740 | Val MAE: 0.3473 | Val CCC: 0.3261 | LR: 1.00e-04
Sem melhora por 10/10 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3917


NUM_LAYERS = 4

In [11]:
NUM_LAYERS = 4

resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=NUM_LAYERS,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=10,
    checkpoint_path=f"/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10.pth",
    unfreeze_epoch=7,
    nome_modelo=f'RESNET50_FINETUNING_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10'
)

Parâmetros treináveis: 17,543,873
TensorBoard: runs/arousal_RESNET50_FINETUNING_NUM_LAYERS_4_PATIENCE_10_20260613-153241
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [02:48<00:00,  1.94it/s]


Época [1/20] [Congelada] | Train Loss: 0.2032 | Val Loss: 0.1840 | Val MAE: 0.3710 | Val CCC: 0.2254 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2254)


Época 2/20 [val]: 100%|██████████| 327/327 [02:48<00:00,  1.94it/s]


Época [2/20] [Congelada] | Train Loss: 0.1942 | Val Loss: 0.1770 | Val MAE: 0.3676 | Val CCC: 0.2787 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2787)


...

Época [3/20] [Congelada] | Train Loss: 0.1828 | Val Loss: 0.1836 | Val MAE: 0.3558 | Val CCC: 0.2738 | LR: 1.00e-04
Sem melhora por 1/10 épocas.


Época 4/20 [val]: 100%|██████████| 327/327 [02:51<00:00,  1.90it/s]


Época [4/20] [Congelada] | Train Loss: 0.1817 | Val Loss: 0.1650 | Val MAE: 0.3427 | Val CCC: 0.3452 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.3452)


Época 5/20 [treino]:  20%|█▉        | 149/757 [01:28<06:00,  1.69it/s]


AttributeError: module 'av' has no attribute 'AVError'

NUM_LAYERS = 5

In [ ]:
NUM_LAYERS = 5

resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=NUM_LAYERS,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=10,
    checkpoint_path=f"/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10.pth",
    unfreeze_epoch=7,
    nome_modelo=f'RESNET50_FINETUNING_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10'
)

Parâmetros treináveis: 19,120,833
TensorBoard: runs/arousal_RESNET50_FINETUNING_NUM_LAYERS_5_20260613-014028
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.87it/s]


Época [1/20] [Congelada] | Train Loss: 0.2092 | Val Loss: 0.1883 | Val MAE: 0.4090 | Val CCC: 0.0833 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0833)


Época 2/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.87it/s]


Época [2/20] [Congelada] | Train Loss: 0.1983 | Val Loss: 0.1827 | Val MAE: 0.3645 | Val CCC: 0.2406 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.2406)


...

Época [10/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1883 | Val Loss: 0.2160 | Val MAE: 0.4049 | Val CCC: 0.1236 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 11/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.87it/s]


Época [11/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1853 | Val Loss: 0.1874 | Val MAE: 0.3799 | Val CCC: 0.2289 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 12/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.86it/s]

Época [12/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1837 | Val Loss: 0.2087 | Val MAE: 0.4125 | Val CCC: 0.0803 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3635


NUM_LAYERS = 6

In [ ]:
NUM_LAYERS = 6

resnet_balanced = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=NUM_LAYERS,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(resnet_balanced.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in resnet_balanced.parameters() if p.requires_grad):,}")

train_model_fine_tuning(
    model=resnet_balanced,
    train_loader=train_video_loader_balanced,
    val_loader=valid_video_loader_balanced,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=10,
    checkpoint_path=f"/mnt/storage_C4/gaussian_football/models/checkpoints/resnet50_lstm_balanced_finetuning_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10.pth",
    unfreeze_epoch=7,
    nome_modelo=f'RESNET50_FINETUNING_NUM_LAYERS_{NUM_LAYERS}_PATIENCE_10'
)

Parâmetros treináveis: 20,697,793
TensorBoard: runs/arousal_RESNET50_FINETUNING_NUM_LAYERS_6_20260613-034952
=> Configuração Inicial: Congelando os pesos da ResNet (exceto a primeira camada)...


Época 1/20 [val]: 100%|██████████| 327/327 [02:55<00:00,  1.87it/s]


Época [1/20] [Congelada] | Train Loss: 0.2009 | Val Loss: 0.1776 | Val MAE: 0.3960 | Val CCC: 0.1373 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1373)


Época 2/20 [val]: 100%|██████████| 327/327 [02:53<00:00,  1.88it/s]


Época [2/20] [Congelada] | Train Loss: 0.1915 | Val Loss: 0.1751 | Val MAE: 0.3845 | Val CCC: 0.1784 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.1784)


...

Época [7/20] [Congelada] | Train Loss: 0.1758 | Val Loss: 0.1713 | Val MAE: 0.3785 | Val CCC: 0.2069 | LR: 1.00e-04
Sem melhora por 3/5 épocas.

[ÉPOCA 8] => ATENÇÃO: Descongelando os pesos da ResNet para Fine-Tuning completo!


Época 8/20 [val]: 100%|██████████| 327/327 [02:54<00:00,  1.87it/s]


Época [8/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1936 | Val Loss: 0.1837 | Val MAE: 0.3805 | Val CCC: 0.2155 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 9/20 [val]: 100%|██████████| 327/327 [02:54<00:00,  1.87it/s]

Época [9/20] [Aberta (Fine-Tuning)] | Train Loss: 0.1918 | Val Loss: 0.1691 | Val MAE: 0.3783 | Val CCC: 0.2198 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.3077
